# Qwen2-VL을 이용한 이미지 이해 실습

### 실습 목표
- **LVLM(Large Vision-Language Model)** 이 무엇인지 이해합니다
- **Qwen2-VL-2B-Instruct** 모델을 Colab에서 직접 불러와 실행해봅니다
- 이미지를 입력으로 주고 모델이 이미지를 어떻게 이해하는지 확인합니다
- 텍스트 질문과 이미지를 함께 넣어 **Visual Question Answering(VQA)** 을 경험합니다

### Contents
1. 환경 설정
2. LVLM이란 무엇인가?
3. Qwen2-VL 모델 불러오기
4. 이미지 설명 생성 (Image Captioning)
5. 이미지 질의응답 (Visual Question Answering)
6. 자유 실습


## 0. 환경 설정

> ⚠️ 반드시 **런타임 > 런타임 유형 변경 > GPU** 로 설정한 후 실습을 진행하세요.

필요한 라이브러리를 설치합니다.

| 라이브러리 | 역할 |
|---|---|
| `transformers` | Hugging Face 모델 로딩 라이브러리 |
| `Pillow` | 이미지 불러오기 및 처리 |
| `requests` | 인터넷에서 이미지 다운로드 |
| `accelerate` | GPU 메모리 효율적 사용 |
| `qwen-vl-utils` | Qwen2-VL 전용 유틸리티 |


In [ ]:
# 필요한 라이브러리 설치
!pip install -q transformers Pillow requests accelerate qwen-vl-utils


In [ ]:
# 라이브러리 불러오기
import torch
import requests
from PIL import Image
from io import BytesIO
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# GPU / CPU 자동 설정
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")


## 1. LVLM이란 무엇인가?

**LVLM(Large Vision-Language Model)** 은 **텍스트와 이미지를 동시에 이해**할 수 있는 대형 AI 모델입니다.

기존 언어 모델(GPT 등)은 텍스트만 처리할 수 있었지만,  
LVLM은 **이미지 + 텍스트**를 함께 입력받아 질문에 답하거나 이미지를 설명할 수 있습니다.

```
입력:  [이미지] + "이 이미지에서 무엇이 보이나요?"
출력:  "이미지에는 고양이 한 마리가 소파 위에 앉아 있습니다."
```

---

### 오늘 사용할 모델: Qwen2-VL-2B-Instruct

| 항목 | 내용 |
|---|---|
| 개발사 | Alibaba (알리바바) |
| 파라미터 수 | 약 2B (20억) |
| 특징 | 이미지 + 텍스트 동시 처리, 한국어 부분 지원 |
| Colab 무료 GPU | ✅ 사용 가능 (VRAM ~4GB) |

2B 모델은 가볍기 때문에 Colab 무료 환경(T4 GPU)에서도 안정적으로 동작합니다.


## 2. Qwen2-VL 모델 불러오기

GPT2에서 `GPT2Tokenizer`와 `GPT2LMHeadModel`을 각각 불러왔던 것처럼,  
Qwen2-VL도 **Processor**와 **Model**을 각각 불러옵니다.

| 구성요소 | 역할 |
|---|---|
| `AutoProcessor` | 이미지와 텍스트를 모델 입력 형태로 변환 (Tokenizer + 이미지 전처리 통합) |
| `Qwen2VLForConditionalGeneration` | 이미지 + 텍스트를 입력받아 텍스트를 생성하는 모델 |

> 💡 처음 실행 시 모델 파일을 다운로드하기 때문에 **1~3분** 정도 소요됩니다.


In [ ]:
# ✅ 1. 모델과 Processor 불러오기
model_name = "Qwen/Qwen2-VL-2B-Instruct"

print("Processor 불러오는 중...")
processor = AutoProcessor.from_pretrained(model_name)

print("모델 불러오는 중... (1~3분 소요)")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # 메모리 절약을 위해 16-bit 사용
    device_map="auto"            # GPU/CPU 자동 배치
)
model.eval()

print("\n✅ 모델 로딩 완료!")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")


## 3. 이미지 불러오기

실습에 사용할 이미지를 인터넷에서 불러옵니다.  
URL 주소만 바꾸면 원하는 이미지를 사용할 수 있습니다.


In [ ]:
# ✅ 2. 이미지 불러오기
def load_image_from_url(url):
    """URL에서 이미지를 불러오는 함수"""
    response = requests.get(url)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    return image

# 샘플 이미지 (뉴욕 시내 사진)
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/New_york_times_square-terabass.jpg/1200px-New_york_times_square-terabass.jpg"

image = load_image_from_url(IMAGE_URL)

# 이미지 크기 확인 및 시각화
print(f"이미지 크기: {image.size}  (가로 x 세로)")
image


## 4. 이미지 설명 생성 (Image Captioning)

이미지를 입력으로 주고, 모델이 이미지를 설명하도록 합니다.

### 동작 흐름

```
[이미지] + [텍스트 질문]
   ↓
Processor (이미지 + 텍스트를 모델 입력 형태로 변환)
   ↓
Qwen2-VL 모델
   ↓
생성된 텍스트 답변
```


In [ ]:
# ✅ 3. 이미지 설명 생성 함수 정의
def generate_response(image, question, max_new_tokens=256):
    """
    image        : PIL Image 객체
    question     : 모델에게 물어볼 질문 (텍스트)
    max_new_tokens: 생성할 최대 토큰 수
    """
    # 메시지 형식 구성 (Qwen2-VL 형식)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  question},
            ],
        }
    ]

    # Processor로 입력 준비
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    # 모델로 텍스트 생성
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    # 입력 부분 제거 후 디코딩
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]

print("✅ 함수 정의 완료!")


In [ ]:
# ✅ 4. 이미지 설명 생성 (Image Captioning)
question = "Describe this image in detail."

print(f"질문: {question}")
print("모델 응답 생성 중...")
print("-" * 50)

response = generate_response(image, question)
print(response)


## 5. 이미지 질의응답 (Visual Question Answering)

이미지에 대해 구체적인 질문을 던지고 답변을 받아봅니다.  
아래 셀을 하나씩 실행하면서 결과를 확인해보세요.


In [ ]:
# ✅ 5. VQA - 특정 사물 묻기
question = "What are people doing in this image?"

print(f"질문: {question}")
print("-" * 50)
print(generate_response(image, question))


In [ ]:
# ✅ 6. VQA - 장소 파악하기
question = "Where do you think this photo was taken? Give me specific clues."

print(f"질문: {question}")
print("-" * 50)
print(generate_response(image, question))


In [ ]:
# ✅ 7. VQA - 색상 묻기
question = "What colors are most prominent in this image?"

print(f"질문: {question}")
print("-" * 50)
print(generate_response(image, question))


In [ ]:
# ✅ 8. VQA - 감정/분위기 파악
question = "How would you describe the atmosphere or mood of this scene?"

print(f"질문: {question}")
print("-" * 50)
print(generate_response(image, question))


## 6. 자유 실습 🚀

이미지 URL과 질문을 자유롭게 바꿔보세요!

### 이미지 예시

| 주제 | URL |
|---|---|
| 고양이 | `https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/1200px-Dog_Breeds.jpg` |
| 음식 | `https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg` |
| 자연 | 원하는 이미지의 URL을 붙여넣기 |

### 질문 예시

- `"What objects can you see in this image?"`
- `"Is there any text visible? If so, what does it say?"`
- `"Count the number of people in the image."`
- `"What time of day does this appear to be?"`


In [ ]:
# ✅ [자유 실습] 이미지 URL과 질문을 바꿔보세요!

# ── 여기를 수정하세요 ──────────────────────────────────────
MY_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/New_york_times_square-terabass.jpg/1200px-New_york_times_square-terabass.jpg"
MY_QUESTION  = "What is the most interesting thing about this image?"
# ──────────────────────────────────────────────────────────

# 이미지 불러오기
my_image = load_image_from_url(MY_IMAGE_URL)
print(f"이미지 크기: {my_image.size}")
display(my_image)

# 모델 응답 생성
print(f"\n질문: {MY_QUESTION}")
print("-" * 50)
print(generate_response(my_image, MY_QUESTION))
